In [1]:
from pathlib import Path
from datetime import datetime
import sys
import time
import json
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path("/home/harielpadillasanchez/Documentos/TT/TT2")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUT_DIR = PROJECT_ROOT / "outputs" / "prompting_on_sample36"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE36_CSV = PROJECT_ROOT / "outputs" / "final_sample_36" / "feina_repr30_test_sample36_representative.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SAMPLE36_CSV:", SAMPLE36_CSV, SAMPLE36_CSV.exists())
print("OUT_DIR:", OUT_DIR)

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
SAMPLE36_CSV: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/final_sample_36/feina_repr30_test_sample36_representative.csv True
OUT_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompting_on_sample36


In [3]:
from configs.models import MODELS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df_sample36 = pd.read_csv(SAMPLE36_CSV)

print("Shape df_sample36:", df_sample36.shape)
display(df_sample36.head(3))
print(df_sample36.columns.tolist())

Shape df_sample36: (36, 32)


,row_id,idFinal,source_text,reference_text,idcod,atr0,atr1,atr2,atr3,atr4,...,src_words,src_sentences,has_number,has_money,has_percent,n_rules_bucket,length_bucket,lex_bucket,stratum,split_stratum
0,550,642_LibroUAC_Sincopyright.pdf,"No tiene incentivos ni motivación, dos element...",No tiene sentido ni motivación. Estos elemento...,Fiorella,16.0,6.0,NaN,NaN,NaN,...,12,1,0,0,0,r2,short,0,fam=reduction | rules=r2 | len=short | num=0 |...,fam=reduction | rules=r2 | len=short
1,571,677_LibroUAC_Sincopyright.pdf,En los procesos de importaciones y exportacion...,Clientes y proveedores intervienen en las impo...,Fiorella,15.0,19.0,16.0,6.0,9.0,...,27,1,0,0,0,r4plus,long,0,fam=structural | rules=r4plus | len=long | num...,fam=structural | rules=r4plus | len=long
2,885,17229_LibroBAC.pdf,El robo de identidad ocurre cuando su informac...,El robo de identidad ocurre cuando su informac...,María Annette,9.0,NaN,NaN,NaN,NaN,...,33,1,0,0,0,r1,long,0,fam=morphosyntactic | rules=r1 | len=long | nu...,fam=morphosyntactic | rules=r1 | len=long


['row_id', 'idFinal', 'source_text', 'reference_text', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex', 'rules_list', 'n_rules', 'main_rule', 'main_rule_name', 'families_list', 'n_families', 'main_family', 'src_words', 'src_sentences', 'has_number', 'has_money', 'has_percent', 'n_rules_bucket', 'length_bucket', 'lex_bucket', 'stratum', 'split_stratum']


In [5]:
required_cols = ["row_id", "source_text", "reference_text"]
missing = [c for c in required_cols if c not in df_sample36.columns]
if missing:
    raise ValueError(f"Faltan columnas en df_sample36: {missing}")

print("OK columnas base")

OK columnas base


In [6]:
FEW_SHOT_EXAMPLES = [
    {
        "source": """A continuación una tabla donde se demuestra como crecen los ahorros al pasar los años: Edad Inversión Intereses Saldo Inversión Intereses Saldo Laura Ganados Alberto Ganados 22 $800 $80 $880 23 $800 $168 $1.232 Valor al Jubilarse $311.092 Valor al Jubilarse $263.232 Menos las contribuciones iniciales $6.400 Menos las contribuciones iniciales $28.800 Ganancia Neta $304.692 Ganancia Neta $234.""",
        "target": """A continuación, presentamos una tabla donde se demuestra como crecen los ahorros al pasar los años. Las categorías en que se divide la tabla son edad, inversión, intereses ganados y saldo. La tabla contrasta las ganancias en intereses por año de Laura y Alberto con una proyección de edad de jubilación de sesenta y cinco años y según la cantidad de años que mantuvieron el ahorro."""
    },
    {
        "source": """El varias veces mencionado Yager, nos dice acertadamente sobre estos aspectos lo siguiente: La mayoría de ustedes están hambrientos de tener libertad financiera, están hastiados de tener batallas financieras porque son como un carrusel.""",
        "target": """Yager habla acertadamente sobre estos aspectos y dice que la mayoría ambiciona tener libertad financiera y está cansada de tener batallas financieras inútiles."""
    },
    {
        "source": """De acuerdo con esta medición, la proporción de pobres en la población mundial quienes viven con menos de un dólar por día descendió levemente entre 1987 y 1993, pues pasó del 30% al 29%.""",
        "target": """Según esta medición, la proporción de personas pobres en la población mundial descendió un poco entre mil novecientos ochenta y siete y mil novecientos noventa y tres. Es decir, el porcentaje de personas que vivían con menos de un dólar por día pasó del treinta al veintinueve por ciento."""
    },
    {
        "source": """- Rentabilidad
INTERÉS DE 2 PESOS SOBRE UNA INVERSIÓN DE 100 PESOS = RENTABILIDAD DE 2 % ((2 ÷ 100) × 100 = 2%) POR CADA 100 PESOS SE GANAN 2 PESOS
Endeudamiento
DEUDA DE 30 PESOS POR SOBRE EL CAPITAL DE 20 PESOS = ENDEUDAMIENTO DE 1,5 VECES EL CAPITAL (30 ÷ 20 = 1,5) POR CADA PESO DE CAPITAL EXISTE 1,5 PESOS DE DEUDA
Liquidez
80 PESOS DE DINERO DISPONIBLE POR SOBRE DEUDAS DE 40 PESOS = LIQUIDEZ DE DOS VECES EL VALOR DE LA DEUDA (80 ÷ 40 = 2) POR CADA PESO DE DEUDA SE DISPONE DE 2 PESOS PARA PAGO""",
        "target": """EL INTERÉS DE DOS PESOS SOBRE UNA INVERSIÓN DE CIEN PESOS DA UNA RENTABILIDAD DE DOS POR CIENTO. ES DECIR, POR CADA CIEN PESOS GANAMOS DOS PESOS. LA DEUDA DE TREINTA PESOS SOBRE EL CAPITAL DE VEINTE PESOS RESULTA EN UNA DEUDA DE UNO PUNTO CINCO VECES EL CAPITAL. ES DECIR, POR CADA PESO DE CAPITAL HAY UNO PUNTO CINCO PESOS DE DEUDA. EN OCHENTA PESOS DE DINERO DISPONIBLE SOBRE DEUDAS DE CUARENTA PESOS DA UNA LIQUIDEZ DEL DOBLE DE LA DEUDA. POR CADA PESO ADEUDADO HAY DOS PESOS PARA PAGAR."""
    },
]

FEW_SHOT_EXAMPLE_IDS = [78, 1805, 3635, 5262]

print("Numero de ejemplos few-shot:", len(FEW_SHOT_EXAMPLES))
print("IDs few-shot:", FEW_SHOT_EXAMPLE_IDS)

Numero de ejemplos few-shot: 4
IDs few-shot: [78, 1805, 3635, 5262]


In [7]:
PROMPT_FINALISTS = [
    {
        "prompt_family": "few-shot",
        "owner": "hariel",
        "model_key": "llama3",
        "config_label": "llama3_cfg_3",
        "ruleset": "R1",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 256,
        "few_shot_examples": FEW_SHOT_EXAMPLES,
        "few_shot_example_ids": FEW_SHOT_EXAMPLE_IDS,
    },
    {
        "prompt_family": "few-shot",
        "owner": "nancy",
        "model_key": "llama3",
        "config_label": "llama3_cfg_2",
        "ruleset": "R2",
        "temperature": 0.7,
        "top_p": 0.90,
        "repetition_penalty": 1.10,
        "max_new_tokens": 512,
        "few_shot_examples": FEW_SHOT_EXAMPLES,
        "few_shot_example_ids": FEW_SHOT_EXAMPLE_IDS,
    },
    {
        "prompt_family": "zero-shot",
        "owner": "hariel",
        "model_key": "llama3",
        "config_label": "hariel_llama3_cfg_1",
        "ruleset": "R0",
        "temperature": 0.2,
        "top_p": 0.85,
        "repetition_penalty": 1.05,
        "max_new_tokens": 256,
        "few_shot_examples": None,
        "few_shot_example_ids": [],
    },
    {
        "prompt_family": "zero-shot",
        "owner": "nancy",
        "model_key": "mistral",
        "config_label": "nancy_mistral_cfg_2",
        "ruleset": "R4",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 400,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "few_shot_examples": None,
        "few_shot_example_ids": [],
    },
]

display(pd.DataFrame(PROMPT_FINALISTS))

,prompt_family,owner,model_key,config_label,ruleset,temperature,top_p,repetition_penalty,max_new_tokens,few_shot_examples,few_shot_example_ids,do_sample,no_repeat_ngram_size
0,few-shot,hariel,llama3,llama3_cfg_3,R1,0.3,0.90,1.15,256,[{'source': 'A continuación una tabla donde se...,"[78, 1805, 3635, 5262]",NaN,NaN
1,few-shot,nancy,llama3,llama3_cfg_2,R2,0.7,0.90,1.10,512,[{'source': 'A continuación una tabla donde se...,"[78, 1805, 3635, 5262]",NaN,NaN
2,zero-shot,hariel,llama3,hariel_llama3_cfg_1,R0,0.2,0.85,1.05,256,None,[],NaN,NaN
3,zero-shot,nancy,mistral,nancy_mistral_cfg_2,R4,0.3,0.90,1.15,400,None,[],True,4.0


In [8]:
for cfg in PROMPT_FINALISTS:
    if cfg["model_key"] not in MODELS:
        raise ValueError(f"Modelo no definido en MODELS: {cfg['model_key']}")

print("Configuraciones validadas.")

Configuraciones validadas.


In [9]:
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_prompting_sample36_top4_{RUN_TS}"

runner = ExperimentRunner(
    experiment_id=EXPERIMENT_ID,
    log_dir=str(PROJECT_ROOT / "outputs" / "logs")
)

print("EXPERIMENT_ID:", EXPERIMENT_ID)

EXPERIMENT_ID: exp_prompting_sample36_top4_20260505_022729


In [10]:
test_row = df_sample36.iloc[0]
test_cfg = PROMPT_FINALISTS[0]

generation_config_test = {
    "temperature": test_cfg["temperature"],
    "top_p": test_cfg["top_p"],
    "repetition_penalty": test_cfg["repetition_penalty"],
    "max_new_tokens": test_cfg["max_new_tokens"],
}

if "do_sample" in test_cfg:
    generation_config_test["do_sample"] = test_cfg["do_sample"]

if "no_repeat_ngram_size" in test_cfg:
    generation_config_test["no_repeat_ngram_size"] = test_cfg["no_repeat_ngram_size"]

test_record = runner.run_one(
    dataset_name="feina_repr30_test_sample36",
    model_key=test_cfg["model_key"],
    prompt_type=test_cfg["prompt_family"],
    ruleset=test_cfg["ruleset"],
    source_text=test_row["source_text"],
    reference_text=test_row["reference_text"],
    sample_id=str(test_row["row_id"]),
    fold_id=None,
    split_name="sample36",
    few_shot_examples=test_cfg["few_shot_examples"],
    few_shot_example_ids=test_cfg["few_shot_example_ids"],
    generation_config=generation_config_test,
)

test_record.to_dict()

{'experiment_id': 'exp_prompting_sample36_top4_20260505_022729',
 'run_id': 'b336be6c-86ca-4a60-b452-bd0489b0dede',
 'timestamp': '2026-05-05T02:27:29.179992',
 'dataset_name': 'feina_repr30_test_sample36',
 'fold_id': None,
 'split_name': 'sample36',
 'model_key': 'llama3',
 'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'backend': 'ollama',
 'prompt_type': 'few-shot',
 'ruleset': 'R1',
 'few_shot_example_ids': [78, 1805, 3635, 5262],
 'generation_config': {'temperature': 0.3,
  'top_p': 0.9,
  'repetition_penalty': 1.15,
  'max_new_tokens': 256},
 'sample_id': '550',
 'source_text': 'No tiene incentivos ni motivación, dos elementos esenciales en ambientes de competencia.',
 'reference_text': 'No tiene sentido ni motivación. Estos elementos son esenciales para la competencia.',
 'generated_text': 'Sin motivación ni incentivos no se puede trabajar bien en un ambiente competitivo.',
 'prompt_text': 'Reescribe en español cada texto con lenguaje más claro y sencillo.\nConserva el sig

In [11]:
all_records = []

total_runs = len(PROMPT_FINALISTS) * len(df_sample36)
run_counter = 0

t0 = time.perf_counter()

for cfg in PROMPT_FINALISTS:
    for _, row in df_sample36.iterrows():
        run_counter += 1

        generation_config = {
            "temperature": cfg["temperature"],
            "top_p": cfg["top_p"],
            "repetition_penalty": cfg["repetition_penalty"],
            "max_new_tokens": cfg["max_new_tokens"],
        }

        if "do_sample" in cfg:
            generation_config["do_sample"] = cfg["do_sample"]

        if "no_repeat_ngram_size" in cfg:
            generation_config["no_repeat_ngram_size"] = cfg["no_repeat_ngram_size"]

        print(
            f"[{run_counter}/{total_runs}] "
            f"prompt_family={cfg['prompt_family']} | "
            f"owner={cfg['owner']} | "
            f"model={cfg['model_key']} | "
            f"cfg={cfg['config_label']} | "
            f"ruleset={cfg['ruleset']} | "
            f"row_id={row['row_id']}"
        )

        record = runner.run_one(
            dataset_name="feina_repr30_test_sample36",
            model_key=cfg["model_key"],
            prompt_type=cfg["prompt_family"],
            ruleset=cfg["ruleset"],
            source_text=row["source_text"],
            reference_text=row["reference_text"],
            sample_id=str(row["row_id"]),
            fold_id=None,
            split_name="sample36",
            few_shot_examples=cfg["few_shot_examples"],
            few_shot_example_ids=cfg["few_shot_example_ids"],
            generation_config=generation_config,
        )

        rec = record.to_dict()
        rec["row_id"] = row["row_id"]
        rec["prompt_family"] = cfg["prompt_family"]
        rec["owner"] = cfg["owner"]
        rec["config_label"] = cfg["config_label"]
        rec["ruleset"] = cfg["ruleset"]
        rec["gen_temperature"] = cfg["temperature"]
        rec["gen_top_p"] = cfg["top_p"]
        rec["gen_repetition_penalty"] = cfg["repetition_penalty"]
        rec["gen_max_new_tokens"] = cfg["max_new_tokens"]
        rec["gen_do_sample"] = cfg.get("do_sample", np.nan)
        rec["gen_no_repeat_ngram_size"] = cfg.get("no_repeat_ngram_size", np.nan)

        all_records.append(rec)

t1 = time.perf_counter()
print(f"\nTiempo total: {(t1 - t0)/60:.2f} min")
print("Total records:", len(all_records))

[1/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=550
[2/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=571
[3/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=885
[4/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=1193
[5/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=1391
[6/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=1399
[7/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=1636
[8/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=1740
[9/144] prompt_family=few-shot | owner=hariel | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | row_id=1742
[10/144] prompt_family=few-shot | owner=

In [12]:
raw_df = pd.DataFrame(all_records)

print("Shape raw_df:", raw_df.shape)
display(raw_df.head(3))
display(raw_df.columns.tolist())

Shape raw_df: (144, 32)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,row_id,prompt_family,owner,config_label,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,gen_do_sample,gen_no_repeat_ngram_size
0,exp_prompting_sample36_top4_20260505_022729,5da3ab91-74bc-412b-97f1-c4314bf29c85,2026-05-05T02:27:54.697730,feina_repr30_test_sample36,None,sample36,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,550,few-shot,hariel,llama3_cfg_3,0.3,0.9,1.15,256,NaN,NaN
1,exp_prompting_sample36_top4_20260505_022729,ef76b546-02c9-4365-8968-f0d71dbf182c,2026-05-05T02:28:10.286189,feina_repr30_test_sample36,None,sample36,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,571,few-shot,hariel,llama3_cfg_3,0.3,0.9,1.15,256,NaN,NaN
2,exp_prompting_sample36_top4_20260505_022729,ccd5b843-da04-492e-93a3-9fe4689ae854,2026-05-05T02:28:27.880568,feina_repr30_test_sample36,None,sample36,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,885,few-shot,hariel,llama3_cfg_3,0.3,0.9,1.15,256,NaN,NaN


['experiment_id',
 'run_id',
 'timestamp',
 'dataset_name',
 'fold_id',
 'split_name',
 'model_key',
 'model_id',
 'backend',
 'prompt_type',
 'ruleset',
 'few_shot_example_ids',
 'generation_config',
 'sample_id',
 'source_text',
 'reference_text',
 'generated_text',
 'prompt_text',
 'inference_seconds',
 'status',
 'error_message',
 'metrics',
 'row_id',
 'prompt_family',
 'owner',
 'config_label',
 'gen_temperature',
 'gen_top_p',
 'gen_repetition_penalty',
 'gen_max_new_tokens',
 'gen_do_sample',
 'gen_no_repeat_ngram_size']

In [13]:
if "status" in raw_df.columns:
    eval_input_df = raw_df[raw_df["status"] == "success"].copy()
else:
    eval_input_df = raw_df.copy()

print("Shape eval_input_df:", eval_input_df.shape)

evaluated_df = evaluate_dataframe(
    eval_input_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
    compute_bertscore=True,
    compute_sbert=False,
)

print("Shape evaluated_df:", evaluated_df.shape)
display(evaluated_df.head(3))

Shape eval_input_df: (144, 32)
Shape evaluated_df: (144, 57)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_prompting_sample36_top4_20260505_022729,5da3ab91-74bc-412b-97f1-c4314bf29c85,2026-05-05T02:27:54.697730,feina_repr30_test_sample36,None,sample36,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.714286,0.666667,63.785000,39.085000,24.700000,0.214286,0.076923,0.142857,0.744898,None
1,exp_prompting_sample36_top4_20260505_022729,ef76b546-02c9-4365-8968-f0d71dbf182c,2026-05-05T02:28:10.286189,feina_repr30_test_sample36,None,sample36,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.500000,0.703704,38.978750,41.390556,-2.411806,0.454545,0.190476,0.272727,0.785588,None
2,exp_prompting_sample36_top4_20260505_022729,ccd5b843-da04-492e-93a3-9fe4689ae854,2026-05-05T02:28:27.880568,feina_repr30_test_sample36,None,sample36,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.379310,0.454545,51.086724,47.347121,3.739603,0.617647,0.303030,0.588235,0.852907,None


In [20]:
group_df = evaluated_df.copy()

group_df["gen_do_sample"] = group_df["gen_do_sample"].astype("string").fillna("NA")
group_df["gen_no_repeat_ngram_size"] = group_df["gen_no_repeat_ngram_size"].astype("string").fillna("NA")

group_cols = [
    "prompt_family",
    "owner",
    "model_key",
    "config_label",
    "ruleset",
    "gen_temperature",
    "gen_top_p",
    "gen_repetition_penalty",
    "gen_max_new_tokens",
    "gen_do_sample",
    "gen_no_repeat_ngram_size",
]

summary_df = summarize_metrics(
    group_df,
    group_cols=group_cols,
)

summary_df["leader_score"] = (
    0.6 * summary_df["sari"] +
    0.4 * (summary_df["bertscore_f1"] * 100.0)
)

display(
    summary_df[
        [
            "prompt_family",
            "owner",
            "model_key",
            "config_label",
            "ruleset",
            "sari",
            "bertscore_f1",
            "leader_score",
            "rougeL_f",
            "compression_ratio_eval",
            "exact_copy",
        ]
    ].sort_values(
        by=["leader_score", "sari", "bertscore_f1"],
        ascending=False,
    )
)

,prompt_family,owner,model_key,config_label,ruleset,sari,bertscore_f1,leader_score,rougeL_f,compression_ratio_eval,exact_copy
0,few-shot,hariel,llama3,llama3_cfg_3,R1,36.161632,0.814534,54.278349,0.402859,1.040341,0.0
1,few-shot,nancy,llama3,llama3_cfg_2,R2,35.025034,0.818844,53.768799,0.389381,0.923332,0.0
2,zero-shot,hariel,llama3,hariel_llama3_cfg_1,R0,34.580136,0.813240,53.277701,0.387453,0.763318,0.0
3,zero-shot,nancy,mistral,nancy_mistral_cfg_2,R4,36.577888,0.773919,52.903505,0.380849,1.261758,0.0


In [21]:
ranking_df = summary_df.sort_values(
    by=["leader_score", "sari", "bertscore_f1"],
    ascending=False
).reset_index(drop=True)

display(ranking_df)

,prompt_family,owner,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,gen_do_sample,...,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1,leader_score
0,few-shot,hariel,llama3,llama3_cfg_3,R1,0.3,0.90,1.15,256,NA,...,0.502279,0.584855,0.447907,0.246218,0.402859,61.141883,59.599136,1.542747,0.814534,54.278349
1,few-shot,nancy,llama3,llama3_cfg_2,R2,0.7,0.90,1.10,512,NA,...,0.506607,0.592203,0.446025,0.227822,0.389381,66.487015,59.599136,6.887879,0.818844,53.768799
2,zero-shot,hariel,llama3,hariel_llama3_cfg_1,R0,0.2,0.85,1.05,256,NA,...,0.458538,0.609197,0.441423,0.229333,0.387453,64.030230,59.599136,4.431094,0.813240,53.277701
3,zero-shot,nancy,mistral,nancy_mistral_cfg_2,R4,0.3,0.90,1.15,400,True,...,0.574438,0.540120,0.421734,0.212082,0.380849,65.673031,59.599136,6.073894,0.773919,52.903505


In [22]:
simplified_texts_df = evaluated_df[
    [
        c for c in [
            "row_id",
            "prompt_family",
            "owner",
            "model_key",
            "config_label",
            "ruleset",
            "source_text",
            "reference_text",
            "generated_text",
            "sari",
            "bertscore_f1",
            "rougeL_f",
            "compression_ratio_eval",
            "exact_copy",
        ] if c in evaluated_df.columns
    ]
].copy()

simplified_texts_df = simplified_texts_df.sort_values(
    by=["prompt_family", "owner", "model_key", "config_label", "row_id"]
).reset_index(drop=True)

display(simplified_texts_df.head(12))

,row_id,prompt_family,owner,model_key,config_label,ruleset,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
0,550,few-shot,hariel,llama3,llama3_cfg_3,R1,"No tiene incentivos ni motivación, dos element...",No tiene sentido ni motivación. Estos elemento...,Sin motivación y sin incentivos no se puede tr...,32.203642,0.744898,0.142857,1.166667,0
1,571,few-shot,hariel,llama3,llama3_cfg_3,R1,En los procesos de importaciones y exportacion...,Clientes y proveedores intervienen en las impo...,"En la importación y exportación, también parti...",44.060676,0.785588,0.272727,0.592593,0
2,885,few-shot,hariel,llama3,llama3_cfg_3,R1,El robo de identidad ocurre cuando su informac...,El robo de identidad ocurre cuando su informac...,El robo de identidad es cuando alguien utiliza...,17.219228,0.852907,0.588235,0.878788,0
3,1193,few-shot,hariel,llama3,llama3_cfg_3,R1,Los sueños de los niños tienen una gran varied...,Los sueños de los niños son variados y van des...,"Los niños soñan con cosas muy diferentes, como...",39.355610,0.837914,0.473684,0.750000,0
4,1391,few-shot,hariel,llama3,llama3_cfg_3,R1,Expresiones como soborno y cohecho se refieren...,Expresiones como soborno y cohecho se refieren...,"Expresiones como ""soborno"" y ""cohecho"" se refi...",39.102538,0.895930,0.585366,1.000000,0
5,1399,few-shot,hariel,llama3,llama3_cfg_3,R1,"DIRECCIÓN DE PRESUPUESTO (DIPRES): ""formula y ...",DIRECCIÓN DE PRESUPUESTO: formula y lleva el c...,La Dirección de Presupuesto es responsable de ...,31.316305,0.706406,0.333333,1.076923,0
6,1636,few-shot,hariel,llama3,llama3_cfg_3,R1,Interés compuesto: Es la suma del capital y el...,El interés compuesto es la suma del capital y ...,El interés compuesto es la suma del capital y ...,56.657226,0.940141,0.828571,0.942857,0
7,1740,few-shot,hariel,llama3,llama3_cfg_3,R1,La producción en un país se realiza mediante l...,A través de los factores de producción se real...,La producción en un país se hace utilizando re...,45.353193,0.788121,0.355556,0.538462,0
8,1742,few-shot,hariel,llama3,llama3_cfg_3,R1,A pesar de que algunos critican al intermediar...,Aunque algunas personas critican al intermedia...,Aunque algunos critican al intermediario finan...,56.211569,0.882987,0.642857,0.812500,0
9,1790,few-shot,hariel,llama3,llama3_cfg_3,R1,El impuesto a los réditos es también un buen i...,El impuesto a las utilidades también es un bue...,El impuesto sobre los intereses es una forma e...,39.711278,0.890636,0.478261,0.954545,0


In [23]:
raw_path = OUT_DIR / f"{EXPERIMENT_ID}_raw_results.csv"
eval_path = OUT_DIR / f"{EXPERIMENT_ID}_evaluated.csv"
summary_path = OUT_DIR / f"{EXPERIMENT_ID}_summary.csv"
ranking_path = OUT_DIR / f"{EXPERIMENT_ID}_ranking.csv"
simplified_texts_path = OUT_DIR / f"{EXPERIMENT_ID}_simplified_texts_only.csv"

raw_df.to_csv(raw_path, index=False, encoding="utf-8-sig")
evaluated_df.to_csv(eval_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
ranking_df.to_csv(ranking_path, index=False, encoding="utf-8-sig")
simplified_texts_df.to_csv(simplified_texts_path, index=False, encoding="utf-8-sig")

print("Guardado raw            :", raw_path)
print("Guardado evaluated      :", eval_path)
print("Guardado summary        :", summary_path)
print("Guardado ranking        :", ranking_path)
print("Guardado simplified csv :", simplified_texts_path)

Guardado raw            : /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompting_on_sample36/exp_prompting_sample36_top4_20260505_022729_raw_results.csv
Guardado evaluated      : /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompting_on_sample36/exp_prompting_sample36_top4_20260505_022729_evaluated.csv
Guardado summary        : /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompting_on_sample36/exp_prompting_sample36_top4_20260505_022729_summary.csv
Guardado ranking        : /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompting_on_sample36/exp_prompting_sample36_top4_20260505_022729_ranking.csv
Guardado simplified csv : /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompting_on_sample36/exp_prompting_sample36_top4_20260505_022729_simplified_texts_only.csv


In [24]:
metadata = {
    "experiment_id": EXPERIMENT_ID,
    "sample_size": int(len(df_sample36)),
    "n_candidates": int(len(PROMPT_FINALISTS)),
    "sample_csv": str(SAMPLE36_CSV),
    "raw_path": str(raw_path),
    "eval_path": str(eval_path),
    "summary_path": str(summary_path),
    "ranking_path": str(ranking_path),
    "simplified_texts_path": str(simplified_texts_path),
}

meta_path = OUT_DIR / f"{EXPERIMENT_ID}_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Guardado metadata:", meta_path)

Guardado metadata: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompting_on_sample36/exp_prompting_sample36_top4_20260505_022729_metadata.json


In [25]:
print("Resumen final sample36 prompting")
print("-" * 50)
print("Total filas evaluadas:", len(evaluated_df))
print("Total simplificaciones guardadas:", len(simplified_texts_df))

best_row = ranking_df.iloc[0]

for col in [
    "prompt_family",
    "owner",
    "model_key",
    "config_label",
    "ruleset",
    "sari",
    "bertscore_f1",
    "leader_score",
]:
    if col in best_row.index:
        print(f"{col}: {best_row[col]}")

Resumen final sample36 prompting
--------------------------------------------------
Total filas evaluadas: 144
Total simplificaciones guardadas: 144
prompt_family: few-shot
owner: hariel
model_key: llama3
config_label: llama3_cfg_3
ruleset: R1
sari: 36.1616321438465
bertscore_f1: 0.8145342336760627
leader_score: 54.278348633350404
